# 080. Warsztatowy Case Study: 130 Lat Letnich Igrzysk Olimpijskich (1896–2024)

Kompleksowy projekt end-to-end realizowany na autentycznych danych z bazy Letnich Igrzysk Olimpijskich (z archiwum `../olimpics.zip`).

> **Higiena pracy analityka:**
> Prawdziwy analityk danych nie boi się plików ZIP – rozpakowanie archiwum (`unzip olimpics.zip`), zmiana nazwy czy obsługa formatów kompresji to chleb powszedni. Pandas potrafi również wczytać archiwum ZIP bezpośrednio w locie (`pd.read_csv("../olimpics.zip")`)!


In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
# KROK 1: Wczytanie danych z archiwum ZIP lub rozpakowanego pliku CSV
if os.path.exists("../olimpics.zip"):
    df_raw = pd.read_csv("../olimpics.zip")
elif os.path.exists("olympics_dataset.csv"):
    df_raw = pd.read_csv("olympics_dataset.csv")
else:
    df_raw = pd.read_csv("olympics.csv")


In [ ]:
# Ujednolicenie nazw kolumn i odfiltrowanie pozycji bez medalu
df_olimp = df_raw.rename(columns={"Name": "Athlete", "Team": "Country"}).copy()
if "Medal" in df_olimp.columns and "No medal" in df_olimp["Medal"].values:
    df_olimp = df_olimp[df_olimp["Medal"] != "No medal"]


In [ ]:
# Optymalizacja pamięciowa: typ kategoryczny dla powtarzalnych napisów
for col in ["City", "Sport", "Event", "Country", "Medal"]:
    if col in df_olimp.columns:
        df_olimp[col] = df_olimp[col].astype("category")


In [ ]:
# Wyniki
print("Wymiary tabeli:", df_olimp.shape)
print("Kolumny:", df_olimp.columns.tolist())
print("\nLiczba unikalnych sportowców:", df_olimp["Athlete"].nunique())
print("Liczba krajów:", df_olimp["Country"].nunique())


## 2. Analiza równouprawnienia płci (Gender Diversity) w historii igrzysk

Jak zmieniał się udział kobiet wśród medalistów olimpijskich od początku ery nowożytnej aż do Paryża 2024?


In [ ]:
# Zliczamy medalistów w rozbiciu na płeć w poszczególnych edycjach
medalisci_plec = df_olimp.groupby(["Year", "Sex"]).size().unstack(fill_value=0)

# Obliczenie udziału procentowego kobiet
medalisci_plec["Procent_Kobiet"] = (
    medalisci_plec["F"] / (medalisci_plec["F"] + medalisci_plec["M"]) * 100
).round(1)

print("Próbka udziału kobiet na podium w wybranych edycjach:")
print(medalisci_plec.loc[[1900, 1924, 1976, 2000, 2024], ["F", "M", "Procent_Kobiet"]])

# Wykres trendu równouprawnienia
fig_gender = px.line(
    medalisci_plec.reset_index(),
    x="Year",
    y="Procent_Kobiet",
    markers=True,
    title="Udział procentowy kobiet wśród medalistów Letnich Igrzysk Olimpijskich (1896-2024)",
    labels={"Year": "Rok Igrzysk", "Procent_Kobiet": "% Kobiet na podium"},
    template="plotly_white"
)
fig_gender.show()


## 3. Ważony ranking medalowy sportowców (Złoto=6, Srebro=2, Brąz=1)

Samo sumowanie medali jest niesprawiedliwe dla wielokrotnych mistrzów olimpijskich. Zastosujmy punktację ważoną: $6 \times \text{Złoto} + 2 \times \text{Srebro} + 1 \times \text{Brąz}$.


In [ ]:
# Tabela: Sportowiec x Kolor Medalu
sportowcy_medale = df_olimp.pivot_table(
    index=["Athlete", "Country", "Sport"],
    columns="Medal",
    values="Year",
    aggfunc="count",
    fill_value=0
)[["Gold", "Silver", "Bronze"]]

# Wyliczenie punktów ważonych
sportowcy_medale["Punkty"] = (
    6 * sportowcy_medale["Gold"] + 
    2 * sportowcy_medale["Silver"] + 
    1 * sportowcy_medale["Bronze"]
)
sportowcy_medale["Razem_Medali"] = sportowcy_medale[["Gold", "Silver", "Bronze"]].sum(axis=1)

# Top 10 sportowców wszech czasów
top10 = sportowcy_medale.sort_values(by=["Punkty", "Gold"], ascending=False).head(10)
print("--- TOP 10 NAJBARDZIEJ UTYTUŁOWANYCH SPORTOWCÓW WSZECH CZASÓW ---")
print(top10[["Gold", "Silver", "Bronze", "Razem_Medali", "Punkty"]])


## 4. Wielki Paradoks Medalowy: Medalista vs Medal Kraju (Montreal 1976)

> **Zagadka analityczna:**
> W surowej bazie z igrzysk w Montrealu (1976) na podium stanęło aż **73 Polaków**. Oficjalnie Polska zdobyła jednak **26 medali** (7 złotych, 6 srebrnych, 13 brązowych). Dlaczego?
> **Odpowiedź:** W sportach zespołowych (np. złoto siatkarzy Wagnera) medal otrzymuje 12 zawodników. W bazie danych każdy ma osobny wiersz, lecz dla kraju to JEDEN medal w konkurencji! Usuwamy duplikaty po `subset=["Year", "Country", "Sport", "Event", "Medal"]`.


In [ ]:
# KROK 1: Deduplikacja do poziomu unikalnych konkurencji medalowych kraju
df_oficjalne = df_olimp.drop_duplicates(
    subset=["Year", "Country", "Sport", "Event", "Medal"]
).copy()

# KROK 2: Weryfikacja Montrealu 1976 dla Polski
polska_1976 = df_oficjalne.query("Country in ['POL', 'Poland'] and Year == 1976")
bilans_1976 = polska_1976["Medal"].value_counts()[["Gold", "Silver", "Bronze"]]
print("Oficjalny dorobek medalowy Polski w Montrealu 1976 (po deduplikacji):")
print(bilans_1976)
print("Łączna suma medali:", bilans_1976.sum())  # Dokładnie 26 medali!


## 5. Geopolityka i Bojkoty Olimpijskie (Moskwa 1980 vs Los Angeles 1984)

Sprawdzamy, które państwa zdobyły medale w Moskwie (1980), ale w ramach bojkotu nie pojawiły się na podium w Los Angeles (1984).


In [ ]:
kraje_moskwa = set(df_oficjalne[df_oficjalne["Year"] == 1980]["Country"].unique())
kraje_la = set(df_oficjalne[df_oficjalne["Year"] == 1984]["Country"].unique())

kraje_bojkot = sorted(list(kraje_moskwa - kraje_la))
print("Kraje na podium w Moskwie 1980, które zbojkotowały Igrzyska w LA 1984:")
print(kraje_bojkot[:15])


## 6. Oficjalna tabela medalowa wszech czasów i bilans Polski


In [ ]:
# Tabela medalowa wszech czasów
tabela_medalowa = df_oficjalne.pivot_table(
    index="Country",
    columns="Medal",
    values="Event",
    aggfunc="count",
    fill_value=0
)[["Gold", "Silver", "Bronze"]]
tabela_medalowa["Total"] = tabela_medalowa.sum(axis=1)

print("Top 10 potęg olimpijskich wszech czasów:")
print(tabela_medalowa.sort_values(by=["Gold", "Silver", "Bronze"], ascending=False).head(10))

# Bilans medalowy Polski w poszczególnych edycjach
polska_lata = df_oficjalne.query("Country in ['POL', 'Poland']").pivot_table(
    index="Year",
    columns="Medal",
    values="Event",
    aggfunc="count",
    fill_value=0
)[["Gold", "Silver", "Bronze"]]

fig_pol = px.bar(
    polska_lata.reset_index(),
    x="Year",
    y=["Gold", "Silver", "Bronze"],
    title="Dorobek medalowy Polski na Letnich Igrzyskach Olimpijskich (1896-2024)",
    labels={"Year": "Rok Igrzysk", "value": "Liczba medali", "variable": "Kruszec"},
    color_discrete_map={"Gold": "#FFD700", "Silver": "#C0C0C0", "Bronze": "#CD7F32"},
    barmode="stack",
    template="plotly_white"
)
fig_pol.show()
